
# Olist Logistics SLA & Customer Retention Analytics

## Notebook 00 — Python Environment & Source Data Validation

### 1. Notebook Overview

This notebook establishes the Python analytical environment
and verifies the connection between Python and the SQL Server
database developed for this project.

It retrieves order-level data from the validated SQL Server
staging layer and performs initial structural and data-quality
checks before exploratory data analysis begins.

### 2. Business Context

This project investigates how effectively the Olist marketplace
meets its estimated delivery commitments and explores whether
poor delivery experiences are associated with differences in
subsequent customer ordering behaviour.

Python will be used for exploratory data analysis, data
visualisation and statistical investigation.

### 3. Notebook Objectives

- Verify the Python environment and required libraries.
- Establish a connection to SQL Server.
- Retrieve order-level data from the staging layer.
- Inspect the dataset structure and data types.
- Validate record counts, unique identifiers and missing values.
- Confirm readiness for further analysis.

### 4. Data Source

| Attribute | Description |
|---|---|
| Database | OlistAnalytics |
| Platform | Microsoft SQL Server |
| Schema | stg |
| Table | orders |
| Analytical tool | Python |
| Data manipulation | Pandas |
| Database connectivity | SQLAlchemy and pyodbc |

### 5. Data Handling Approach

The SQL Server staging layer is the trusted source for this
notebook.

Data cleaning, type standardisation and initial data-quality
investigations have already been performed in SQL Server.

This notebook does not repeat those operations. Its purpose
is to verify the successful retrieval and interpretation of
the staged data in Python.

### 6. Expected Outcome

A successfully established Python-to-SQL Server connection,
a correctly retrieved order-level dataset, and documented
validation results before exploratory data analysis begins.



## 1. Environment Configuration

This section verifies the Python environment used for the
project, including the Python version, installation location
and versions of the required analytical libraries.

Recording this information helps maintain consistency and
reproducibility throughout the project.

In [1]:

# ============================================================
# 1. ENVIRONMENT CONFIGURATION
# ============================================================

# Import the built-in system library.
# It provides information about the active Python environment.

import sys

# Display the Python version currently used by Jupyter.
print("Python version:")
print(sys.version)

# Display the location of the Python installation.
# This helps identify which Python environment is running.

print("\nPython executable location:")
print(sys.executable)


# ------------------------------------------------------------
# Check installed analytical library versions
# ------------------------------------------------------------

# importlib.util helps us check whether a library is installed.

import importlib.util

# List the libraries required for this project.

required_libraries = [
    "pandas",
    "numpy",
    "matplotlib",
    "sqlalchemy",
    "pyodbc"
]

# Check the availability of each library.

print("\nLibrary availability:")

for library in required_libraries:

    if importlib.util.find_spec(library) is not None:
        print(f"{library}: Installed")

    else:
        print(f"{library}: Not installed")


Python version:
3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]

Python executable location:
C:\ProgramData\anaconda3\python.exe

Library availability:
pandas: Installed
numpy: Installed
matplotlib: Installed
sqlalchemy: Installed
pyodbc: Installed



## 2. Library Imports

This section imports the Python libraries required for
database connectivity, data manipulation, numerical
calculations and data visualisation.

The libraries are imported at the beginning of the notebook
so they can be reused throughout the subsequent analysis.

In [2]:

# ============================================================
# 2. LIBRARY IMPORTS
# ============================================================

# Pandas: Data manipulation and analysis.
import pandas as pd

# NumPy: Numerical calculations.
import numpy as np

# Matplotlib: Data visualisation.
import matplotlib.pyplot as plt

# SQLAlchemy: Database connectivity.
from sqlalchemy import create_engine, text

# PyODBC: SQL Server database driver.
import pyodbc

# URL encoding: Prepares the SQL Server connection string.
import urllib.parse

print("All required libraries imported successfully.")


All required libraries imported successfully.



## 3. SQL Server Connection

This section establishes a connection between Python and the
Microsoft SQL Server database developed for this project.

The connection uses Windows Authentication, allowing Python
to access the database through the currently authenticated
Windows account without requiring a separate SQL Server password.

SQLAlchemy is used to manage the database connection, while
the Microsoft ODBC Driver provides communication with SQL Server.

**Connection configuration:**

- Server: abhijeet\SQLEXPRESS
- Database: OlistAnalytics
- Authentication: Windows Authentication
- ODBC Driver: ODBC Driver 18 for SQL Server

In [3]:

# ============================================================
# 3. SQL SERVER CONNECTION
# ============================================================

import urllib.parse
from getpass import getpass
from sqlalchemy import create_engine, text

# Ask for the local SQL Server instance name privately.
server = getpass("Enter your SQL Server instance name: ")

database = "OlistAnalytics"
driver = "ODBC Driver 18 for SQL Server"

# Build the SQL Server connection string.
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Convert the connection string for SQLAlchemy.
connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + urllib.parse.quote_plus(connection_string)
)

# Create the database engine.
engine = create_engine(connection_url)

# Test the database connection.
try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT DB_NAME() AS database_name")
        )

        database_name = result.scalar()

        print("Connection successful!")
        print("Connected database:", database_name)

except Exception as error:
    print("Connection failed.")
    print("Error:", error)


Enter your SQL Server instance name:  ········


Connection successful!
Connected database: OlistAnalytics



## 4. Data Extraction

This section retrieves order-level data from the validated
`stg.orders` table in SQL Server.

Only the columns required for the initial logistics performance
analysis are selected.

The extracted data is loaded into a Pandas DataFrame named
`df_orders` for further analysis.

**Selected attributes:**

- order_id
- order_status
- order_purchase_timestamp
- order_delivered_customer_date
- order_estimated_delivery_date

The original SQL Server staging table remains unchanged.

In [4]:

# ============================================================
# 4. DATA EXTRACTION
# ============================================================

# SQL query to retrieve the required order-level attributes.

query = """
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM stg.orders;
"""


# ------------------------------------------------------------
# Execute the SQL query and load the results into Pandas
# ------------------------------------------------------------

# pd.read_sql_query() executes the SQL query and returns
# the retrieved records as a Pandas DataFrame.

df_orders = pd.read_sql_query(
    sql=text(query),
    con=engine
)


# ------------------------------------------------------------
# Display the extracted dataset
# ------------------------------------------------------------

# Display the total number of rows and columns.

print("Dataset successfully retrieved!")
print("Dataset shape:", df_orders.shape)

# Display the first five records.

print("\nFirst five records:")
display(df_orders.head())

# Display the column names.

print("\nColumn names:")
print(df_orders.columns.tolist())

# Display the data types of each column.

print("\nColumn data types:")
print(df_orders.dtypes)


Dataset successfully retrieved!
Dataset shape: (99441, 5)

First five records:


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17



Column names:
['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Column data types:
order_id                                    str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## 5. Structural Validation

This section verifies the structure and integrity of the
extracted order-level dataset.

The validation checks include:

- Total number of records and columns.
- Presence of the expected columns.
- Uniqueness of order identifiers.
- Duplicate order records.
- Data types assigned to each column.

These checks confirm that the extracted dataset has the
expected structure before further analysis.

In [5]:
# ============================================================
# 5. STRUCTURAL VALIDATION
# ============================================================

# 1. Check the dataset dimensions.
# Shape returns two values:
# First value  = Number of rows
# Second value = Number of columns

print("Dataset dimensions:")
print("Total rows:", df_orders.shape[0])
print("Total columns:", df_orders.shape[1])


# 2. Check whether all expected columns are present.

expected_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print("\nExpected columns present:")

for column in expected_columns:

    if column in df_orders.columns:
        print(f"{column}: Present")

    else:
        print(f"{column}: Missing")


# 3. Check the uniqueness of order identifiers.

unique_order_ids = df_orders["order_id"].nunique()

print("\nOrder identifier validation:")
print("Total records:", len(df_orders))
print("Unique order IDs:", unique_order_ids)


# 4. Check for duplicate order identifiers.

duplicate_order_ids = df_orders["order_id"].duplicated().sum()

print("Duplicate order IDs:", duplicate_order_ids)


# 5. Display the data types of all columns.

print("\nColumn data types:")
print(df_orders.dtypes)

Dataset dimensions:
Total rows: 99441
Total columns: 5

Expected columns present:
order_id: Present
order_status: Present
order_purchase_timestamp: Present
order_delivered_customer_date: Present
order_estimated_delivery_date: Present

Order identifier validation:
Total records: 99441
Unique order IDs: 99441
Duplicate order IDs: 0

Column data types:
order_id                                    str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## 6.1 Data Quality Validation

This section examines the completeness and consistency of
the extracted order-level dataset.

The validation focuses on:

- Missing values in each column.
- Distribution of order statuses.
- Missing actual delivery dates across different order statuses.

Missing delivery dates are not automatically considered
data-quality errors. Orders that have not been delivered may
naturally have no actual delivery date.

The results will be compared with the earlier SQL Server
data-quality audit.

In [6]:
# ============================================================
# 6.1 MISSING VALUE ANALYSIS
# ============================================================

# Count missing values in every column.

missing_values = df_orders.isna().sum()

print("Missing values by column:")
print(missing_values)


# Calculate the percentage of missing values in each column.

missing_percentage = (
    df_orders.isna().mean() * 100
).round(2)

print("\nMissing value percentage:")
print(missing_percentage)

Missing values by column:
order_id                            0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing value percentage:
order_id                         0.00
order_status                     0.00
order_purchase_timestamp         0.00
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
dtype: float64


### 6.2 Order Status Distribution

This section examines the distribution of orders across
different operational statuses.

Understanding order statuses helps explain why some orders
do not have an actual delivery timestamp.

The results will also help distinguish completed orders
from cancelled, unavailable and still-in-progress orders.

In [7]:
# ============================================================
# 6.2 ORDER STATUS DISTRIBUTION
# ============================================================

# Count the number of orders in each status.

status_counts = df_orders["order_status"].value_counts()

print("Order status distribution:")
print(status_counts)


# Calculate the percentage represented by each status.

status_percentage = (
    df_orders["order_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nOrder status percentage:")
print(status_percentage)

Order status distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Order status percentage:
order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64


### 6.3 Missing Delivery Dates by Order Status

This section investigates missing actual delivery timestamps
across different order statuses.

The objective is to determine whether missing delivery dates
are associated with non-delivered orders and identify any
delivered orders with missing actual delivery timestamps.

This helps distinguish expected missing values from records
requiring further investigation.

In [8]:
# ============================================================
# 6.3 MISSING DELIVERY DATES BY ORDER STATUS
# ============================================================

# Group orders by their operational status.
# For each status, calculate:
# 1. Total number of orders.
# 2. Number of missing actual delivery dates.
# 3. Number of available actual delivery dates.

delivery_date_status = (
    df_orders
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),

        missing_delivery_dates=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        ),

        available_delivery_dates=(
            "order_delivered_customer_date",
            lambda x: x.notna().sum()
        )
    )
    .sort_values("total_orders", ascending=False)
)

print("Delivery date availability by order status:")
display(delivery_date_status)

Delivery date availability by order status:


,total_orders,missing_delivery_dates,available_delivery_dates
order_status,,,
delivered,96478,8,96470
shipped,1107,1107,0
canceled,625,619,6
unavailable,609,609,0
invoiced,314,314,0
processing,301,301,0
created,5,5,0
approved,2,2,0


In [9]:
# ============================================================
# IDENTIFY DELIVERED ORDERS WITH MISSING DELIVERY DATES
# ============================================================

# Select orders marked as delivered but without
# an actual customer delivery timestamp.

delivered_missing_dates = df_orders[
    (df_orders["order_status"] == "delivered") &
    (df_orders["order_delivered_customer_date"].isna())
]

print(
    "Delivered orders with missing actual delivery dates:",
    len(delivered_missing_dates)
)

display(delivered_missing_dates)

Delivered orders with missing actual delivery dates: 8


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
5168,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT,2018-07-24
12774,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT,2018-07-19
17371,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT,2017-12-18
17548,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,2017-06-23
18041,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT,2018-07-30
66406,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT,2018-06-26
89704,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT,2018-07-30
95490,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT,2018-07-16


### 6.4 Data Quality Findings and Analytical Implications

The missing-value investigation identified 2,965 orders
without an actual customer delivery timestamp.

Further investigation revealed:

- 2,957 non-delivered orders have missing actual delivery dates.
- 8 orders marked as delivered have missing actual delivery dates.
- 6 cancelled orders have available actual delivery timestamps.

The eight delivered orders with missing actual delivery
timestamps cannot be evaluated for delivery timeliness.

Accordingly, these records will be excluded from the
On-Time Delivery Rate calculation, while remaining
preserved in the source dataset.

The six cancelled orders with available delivery timestamps
will also be preserved without automatic correction.

These findings are consistent with the project's established
data-quality and KPI measurement principles.

## 7. Final Validation Summary

### 7.1 Validation Results

The extracted order-level dataset successfully passed the
structural validation checks.

The dataset contains 99,441 records and five expected columns.
All order identifiers are unique, with no duplicate order IDs.

The data-quality investigation identified 2,965 missing
actual delivery timestamps, including eight orders marked
as delivered and six cancelled orders with available
delivery timestamps.

### 7.2 Key Findings

- Total extracted orders: 99,441.
- Unique order identifiers: 99,441.
- Duplicate order identifiers: 0.
- Missing actual delivery timestamps: 2,965.
- Delivered orders with missing actual delivery timestamps: 8.
- Cancelled orders with available actual delivery timestamps: 6.
- Eligible delivered orders for delivery-timeliness analysis: 96,470.

### 7.3 Data Readiness Assessment

The extracted dataset is structurally valid and its
identified data-quality limitations have been documented.

The eight delivered orders without actual delivery
timestamps will be excluded from delivery-timeliness
calculations in accordance with the project KPI Dictionary.

The original extracted dataset remains unchanged.

### 7.4 Conclusion

The Python environment and SQL Server connection have
been successfully established.

The order-level dataset has been retrieved and validated.
The identified limitations are documented and will be
considered during subsequent exploratory data analysis.

Notebook 00 is therefore complete and ready for final
review before the next analytical notebook is developed.

In [10]:
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
